# Phase 0 Supporting Analysis — Prompt-Template Selection for Keyword Embeddings (H1a)

**Policy role.** This is **not** the experiment — it is offline, SONAR-only analysis supporting the
Phase 0 spec: it selects the prompt template (spec §10.1), tests hypothesis **H1a**, and evaluates the
go/no-go pre-check for condition P (spec §7). **No ASR decoding occurs anywhere in this notebook.**

**The question it answers** (research lead's question #1): *how will the keyword embeddings be
computed — and is `"A speech that includes {keyword}"` a good prompt?* Instead of opinions, we
measure. For each candidate template we embed the keyword list and ask how well cosine similarity
against an utterance's SONAR embedding distinguishes:

- **in-context** — keyword embedded, compared against the utterance that *contains* it (gate should
  say YES);
- **out-of-context** — the *same keyword* against utterances that do not contain it (gate should say
  NO — same word, wrong context: isolates the context signal from word identity);
- **distractor** — keywords that never occur in the split at all (the false-alarm population).

**Metrics per template:** mean-similarity gaps, and **ROC-AUC** (threshold-free: the probability that
a random in-context pair outscores a random negative pair; 0.5 = useless gate, 1.0 = perfect).
Exploratory measurement with **bare words** gave a gap of only **0.028** — the number this notebook
tries to beat with templating.

**Oracle caveat (why this is an upper bound):** utterance context here is SONAR's embedding of the
*reference text* — perfect information. At decode time the gate sees the noisier `state @ W`
projection. A template that fails *this* test cannot work at runtime; a template that passes still has
to survive the projection. That directional logic is what makes the diagnostic a valid go/no-go.

**Runtime:** minutes — a few thousand short SONAR embeddings, no audio touched.


## Section 0.1 — Dependencies

In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch datasets pandas matplotlib scikit-learn
# %pip install sonar-space


## Section 0.2 — Imports, seed, config

Line-by-line notes:
- `TEMPLATES` — the candidate set proposed in spec §10.1: bare word (the measured-weak control), the
  research lead's prompt, and three structural variants (mention-framing, topic-framing,
  transcript-framing). Add candidates freely — every downstream cell iterates this dict.
- `tune_frac`/seed replicate **exactly** the Phase 1 shard rule, so the TUNE utterances analyzed here
  are the same utterances Phase 4 will tune on (sharding is by ID only — deliberately independent of
  audio, see Phase 1 §3).
- Caps (`n_utts_cap`, `n_keywords_cap`, ...) bound embedding compute; all selections under the caps
  are deterministic (sorted, then seeded shuffle).


In [ ]:
import json, random, re, itertools
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CONFIG = {
    "splits": {"dev-clean": ("clean", "validation"), "dev-other": ("other", "validation")},
    "tune_frac": 0.25,
    "seed": SEED,
    "min_word_len": 4,
    "max_doc_freq": 3,
    "n_utts_cap": 250,        # TUNE utterances embedded per split
    "n_keywords_cap": 200,    # target keywords per split
    "n_distractors": 100,     # never-occurring keywords per split
    "out_ctx_per_kw": 5,      # sampled wrong-context utterances per keyword
    "go_no_go_gap": 0.05,     # spec §7 threshold (pending lead approval)
}

TEMPLATES = {
    "bare":                "{kw}",
    "lead_speech_includes": 'A speech that includes "{kw}"',
    "speaker_mentions":    "The speaker mentions {kw}",
    "talk_about":          "A recording of someone talking about {kw}",
    "transcript_contains": "An audio transcript that contains the word {kw}",
}

def norm(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print(f"{len(TEMPLATES)} templates:", list(TEMPLATES))


## Section 1 — TUNE references (text only — audio never decoded)

Line-by-line:
1. Stream each split with the **audio column removed** — `remove_columns(["audio"])` guarantees no
   audio bytes are decoded or downloaded beyond the shard manifests; this notebook needs text only.
2. Collect `(id, text)` for the full split, then apply the **identical shard rule as Phase 1 §3**
   (sort IDs → `random.Random(42).shuffle` → first 25% = TUNE). The printed shard SHA must equal
   Phase 1's for the same split — a cheap cross-notebook consistency check.
3. Keep TUNE texts; cap to `n_utts_cap` deterministically (sorted by ID, seeded shuffle, first N).


In [ ]:
from datasets import load_dataset
import hashlib

def sha(obj):
    return hashlib.sha256(json.dumps(sorted(obj)).encode()).hexdigest()[:16]

def shard_ids(ids, tune_frac, seed):
    ordered = sorted(ids)
    random.Random(seed).shuffle(ordered)
    k = int(len(ordered) * tune_frac)
    return set(ordered[:k]), set(ordered[k:])

TUNE_TEXTS = {}
for split_name, (hf_config, hf_split) in CONFIG["splits"].items():
    stream = load_dataset("openslr/librispeech_asr", hf_config,
                          split=hf_split, streaming=True)
    stream = stream.remove_columns(["audio"])
    id_text = [(s["id"], s["text"]) for s in stream]
    tune, _ = shard_ids([i for i, _ in id_text], CONFIG["tune_frac"], CONFIG["seed"])
    rows = sorted([(i, t) for i, t in id_text if i in tune])
    random.Random(CONFIG["seed"]).shuffle(rows)
    TUNE_TEXTS[split_name] = rows[:CONFIG["n_utts_cap"]]
    print(f"{split_name}: {len(id_text)} utts → TUNE {len(tune)} (sha {sha(tune)}) "
          f"→ analyzed {len(TUNE_TEXTS[split_name])}")


## Section 2 — Keywords and distractors per split

Line-by-line:
1. Document frequency over the *analyzed* TUNE texts; **targets** = words ≥ 4 chars, alphabetic,
   df ≤ 3 — the same rarity shape the biasing lists will use — capped deterministically by
   (df, alphabetical).
2. **Distractors** = words passing the same filter in the *other* split's TUNE texts and absent from
   every analyzed utterance of *this* split — guaranteed never-in-context.
3. Pair construction:
   - `in_ctx`: every (keyword, utterance) with the keyword in the utterance;
   - `out_ctx`: per keyword, up to 5 seeded-sampled utterances *not* containing it;
   - `distr`: every distractor × the same 5-per sampling.
   Pair counts printed — the denominators behind every statistic that follows.


In [ ]:
def rare_words(texts, min_len, max_df):
    df = Counter()
    for _, t in texts:
        df.update(set(norm(t).split()))
    ws = [w for w, c in df.items() if len(w) >= min_len and w.isalpha() and c <= max_df]
    return sorted(ws, key=lambda w: (df[w], w)), df

POP = {}
split_names = list(CONFIG["splits"])
for split_name in split_names:
    texts = TUNE_TEXTS[split_name]
    words_sets = [set(norm(t).split()) for _, t in texts]
    targets, _ = rare_words(texts, CONFIG["min_word_len"], CONFIG["max_doc_freq"])
    targets = targets[:CONFIG["n_keywords_cap"]]

    other = [s for s in split_names if s != split_name][0]
    o_targets, _ = rare_words(TUNE_TEXTS[other], CONFIG["min_word_len"], CONFIG["max_doc_freq"])
    vocab = set().union(*words_sets)
    distractors = [w for w in o_targets if w not in vocab][:CONFIG["n_distractors"]]

    rng = random.Random(CONFIG["seed"])
    in_ctx, out_ctx, distr = [], [], []
    for ki, w in enumerate(targets):
        holders = [ui for ui, ws in enumerate(words_sets) if w in ws]
        in_ctx += [(ki, ui) for ui in holders]
        non = [ui for ui in range(len(texts)) if ui not in holders]
        out_ctx += [(ki, ui) for ui in rng.sample(non, min(CONFIG["out_ctx_per_kw"], len(non)))]
    for di in range(len(distractors)):
        distr += [(di, ui) for ui in rng.sample(range(len(texts)),
                                                min(CONFIG["out_ctx_per_kw"], len(texts)))]
    POP[split_name] = {"targets": targets, "distractors": distractors,
                       "in_ctx": in_ctx, "out_ctx": out_ctx, "distr": distr}
    print(f"{split_name}: {len(targets)} targets, {len(distractors)} distractors | "
          f"pairs: in={len(in_ctx)} out={len(out_ctx)} distr={len(distr)}")


## Section 3 — SONAR embeddings

Line-by-line:
1. Utterance references embedded **once per split** (templates only change the keyword side).
2. Per template × split: embed `template.format(kw=w)` for all targets and distractors. All embeddings
   L2-normalized so matrix products are cosines.
3. Total ≈ `2 splits × (250 utts + 5 templates × 300 words)` ≈ 3,500 short texts — minutes on CPU.


In [ ]:
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

t2vec = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                     tokenizer="text_sonar_basic_encoder",
                                     device=torch.device("cpu"))

def embed(texts):
    with torch.no_grad():
        e = t2vec.predict(list(texts), source_lang="eng_Latn", batch_size=64).float()
    return F.normalize(e, dim=-1)

U = {s: embed([norm(t) for _, t in TUNE_TEXTS[s]]) for s in split_names}
print("utterance embeddings:", {s: tuple(U[s].shape) for s in split_names})

KW_EMB = {}
for tname, tmpl in TEMPLATES.items():
    for s in split_names:
        p = POP[s]
        KW_EMB[(tname, s)] = {
            "targets": embed([tmpl.format(kw=w) for w in p["targets"]]),
            "distractors": embed([tmpl.format(kw=w) for w in p["distractors"]]),
        }
    print(f"embedded template: {tname}")


## Section 4 — Scores per template

Line-by-line: for each (template, split), cosine matrices target×utterance and distractor×utterance
are indexed by the three pair populations, yielding three similarity samples; from them:
- `gap_in_distr` = mean(in_ctx) − mean(distr) — the spec §7 go/no-go quantity (bare-word exploratory
  value: 0.028);
- `gap_in_out` = mean(in_ctx) − mean(out_ctx) — pure context signal, word identity held fixed;
- `auc_in_out`, `auc_in_distr` — threshold-free discriminability of the same two contrasts.


In [ ]:
rows = []
SIMS = {}
for tname in TEMPLATES:
    for s in split_names:
        p = POP[s]
        St = (KW_EMB[(tname, s)]["targets"] @ U[s].T).numpy()
        Sd = (KW_EMB[(tname, s)]["distractors"] @ U[s].T).numpy()
        sim_in  = np.array([St[k, u] for k, u in p["in_ctx"]])
        sim_out = np.array([St[k, u] for k, u in p["out_ctx"]])
        sim_dis = np.array([Sd[k, u] for k, u in p["distr"]])
        SIMS[(tname, s)] = (sim_in, sim_out, sim_dis)
        rows.append({
            "template": tname, "split": s,
            "mean_in": sim_in.mean(), "mean_out": sim_out.mean(), "mean_distr": sim_dis.mean(),
            "gap_in_distr": sim_in.mean() - sim_dis.mean(),
            "gap_in_out":   sim_in.mean() - sim_out.mean(),
            "auc_in_out":   roc_auc_score([1]*len(sim_in) + [0]*len(sim_out),
                                          np.concatenate([sim_in, sim_out])),
            "auc_in_distr": roc_auc_score([1]*len(sim_in) + [0]*len(sim_dis),
                                          np.concatenate([sim_in, sim_dis])),
        })

report = pd.DataFrame(rows).round(4)
summary = (report.groupby("template")[["gap_in_distr", "gap_in_out",
                                       "auc_in_out", "auc_in_distr"]]
           .mean().sort_values("auc_in_distr", ascending=False).round(4))
print("=== per split ==="); display(report.sort_values(["template", "split"]))
print("=== averaged over splits (ranking) ==="); summary


## Section 5 — Visual comparison

Left: the go/no-go quantity per template and split, against the 0.05 threshold line and the 0.028
bare-word exploratory value. Right: the three similarity distributions for the **winning** template —
the histogram your research lead should see next to the table.


In [ ]:
best = summary.index[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
piv = report.pivot(index="template", columns="split", values="gap_in_distr").loc[summary.index]
piv.plot.bar(ax=axes[0], rot=20)
axes[0].axhline(CONFIG["go_no_go_gap"], ls="--", c="red", label="go/no-go 0.05")
axes[0].axhline(0.028, ls=":", c="gray", label="bare-word exploratory (0.028)")
axes[0].set_ylabel("mean(in-context) − mean(distractor)")
axes[0].set_title("Gate separation by template"); axes[0].legend()

s0 = split_names[1] if len(split_names) > 1 else split_names[0]
sim_in, sim_out, sim_dis = SIMS[(best, s0)]
axes[1].hist(sim_dis, bins=40, alpha=0.5, density=True, label="distractor")
axes[1].hist(sim_out, bins=40, alpha=0.5, density=True, label="target, wrong utt")
axes[1].hist(sim_in,  bins=40, alpha=0.7, density=True, label="target, own utt")
axes[1].set_xlabel("cosine similarity"); axes[1].set_ylabel("density")
axes[1].set_title(f"Best template: {best!r} on {s0}"); axes[1].legend()
plt.tight_layout(); plt.show()


## Section 6 — Decision output and artifacts

The machine-readable verdict for the spec: winning template, its separation and AUC per split, and
the go/no-go outcome against §7's threshold. `template_diagnostic_report.csv` +
`template_diagnostic_decision.json` are the attachments for the lead's review (with the §5 figure).


In [ ]:
decision = {
    "candidate_templates": TEMPLATES,
    "winner": best,
    "winner_prompt": TEMPLATES[best],
    "per_split": report[report["template"] == best]
                 .set_index("split")[["gap_in_distr", "auc_in_distr", "auc_in_out"]]
                 .round(4).to_dict("index"),
    "go_no_go_threshold": CONFIG["go_no_go_gap"],
    "go_no_go_pass": bool((report[report["template"] == best]["gap_in_distr"]
                           >= CONFIG["go_no_go_gap"]).all()),
    "bare_word_reference": float(summary.loc["bare", "gap_in_distr"]) if "bare" in summary.index else None,
    "config": {k: v for k, v in CONFIG.items() if k != "splits"},
}
report.to_csv("template_diagnostic_report.csv", index=False)
json.dump(decision, open("template_diagnostic_decision.json", "w"), indent=2)
print(json.dumps(decision, indent=2))


## Section 7 — Reading the outcome (maps to spec §7)

- **Winner clears 0.05 on both splits, AUC(in vs distr) ≥ ~0.75:** H1a supported — templating rescued
  the gate signal. Attach table + figure to the spec, fill §10.1 with the winner, and condition P
  proceeds after Phase 1/2. Expect the runtime effect to be *smaller* than this oracle bound.
- **Winner clears 0.05 but AUC is middling (0.6–0.75):** weak-but-real signal. P may proceed, but
  pre-register modest expectations; consider adding stronger context templates (multi-keyword topic
  prompts) to the candidate set *before* freezing the spec.
- **No template clears 0.05:** H1a refuted at word granularity even with oracle context — P is
  cancelled by the go/no-go without spending a single GPU-hour on biased decoding, and the honest
  next spec targets the embedding side (phrase-level biasing entries, topic-list embeddings) or
  non-semantic gating (acoustic-evidence gates). That outcome is a *successful* diagnostic, not a
  failed experiment.

Either way: this notebook, the CSV, the JSON, and the figure are the complete, reproducible answer to
the lead's question #1 — "how you will calculate the embeddings (prompt, e.g.)" — with the selection
made by measurement rather than taste.
